# FASE 3 — Feature Engineering Avanzado

**Objetivo:** Construir y consolidar las variables predictivas estructuradas en los grupos técnicos del proyecto (características de máquina, telemetría original, estadísticas móviles, deltas de tendencia, historial de errores y mantenimiento), respetando estrictamente la consistencia temporal y previniendo el *data leakage*.

*Los hallazgos en la fase de EDA Predictivo guiarán directamente este proceso, documentado en `docs/eda_predictivo.md`*

El notebook procesará el dataset analítico unificado `../data/interim/master_dataset.parquet` y generará la matriz de características procesada para Machine Learning en `../data/processed/features_dataset.parquet`.

In [62]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Estilos visuales
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

> **Resultados y Observación:** Entorno visual e importaciones de manipulación de datos (`pandas`, `numpy`, `os`) configurados correctamente. Librerías de visualización (`matplotlib`, `seaborn`) listas para gráficos de diagnóstico.

## 1. Carga del Master Dataset y Re-validación del Target (Grupo G)

**Justificación:** Cargar `master_dataset.parquet` y verificar el target candidato `failure_next_24h` para confirmar la distribución de clases (1.96% positivos) y asegurar que no exista filtración de información futura.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/No-Country-simulation/S08-26-EQUIPO-24/main/data/interim/master_dataset.parquet"

master_df = pd.read_parquet(url)
print(master_df.shape)

(876100, 19)


In [64]:
# Carga del dataset unificado en local o main
#master_df = pd.read_parquet("../data/interim/master_dataset.parquet")

print("=== INFORMACIÓN GENERAL DEL MASTER DATASET ===")
print(f"Dimensiones: {master_df.shape[0]:,} filas x {master_df.shape[1]} columnas")
print(f"Máquinas únicas: {master_df['machineID'].nunique()}")
print(f"Rango temporal: desde {master_df['datetime'].min()} hasta {master_df['datetime'].max()}")

# Re-verificación del Target
target_counts = master_df['failure_next_24h'].value_counts()
target_pct = master_df['failure_next_24h'].value_counts(normalize=True) * 100

print("\n=== DISTRIBUCIÓN DEL TARGET (failure_next_24h) ===")
print(f"Instancias Clase 0 (Normal): {target_counts[0]:,} ({target_pct[0]:.2f}%)")
print(f"Instancias Clase 1 (Pre-Falla): {target_counts[1]:,} ({target_pct[1]:.2f}%)")
print(f"Razón de desbalance: 1 a {int(target_counts[0]/target_counts[1])}")

=== INFORMACIÓN GENERAL DEL MASTER DATASET ===
Dimensiones: 876,100 filas x 19 columnas
Máquinas únicas: 100
Rango temporal: desde 2015-01-01 06:00:00 hasta 2016-01-01 06:00:00

=== DISTRIBUCIÓN DEL TARGET (failure_next_24h) ===
Instancias Clase 0 (Normal): 858,916 (98.04%)
Instancias Clase 1 (Pre-Falla): 17,184 (1.96%)
Razón de desbalance: 1 a 49


> **Resultados de Carga y Target:** Se cargaron exitosamente las **876,100 filas** de `master_dataset.parquet`. Se re-confirmó la distribución del target `failure_next_24h` (858,916 normales vs 17,184 pre-falla), preservando la alineación temporal sin data leakage.

## 2. Grupo A — Features de Máquina (No Duplicadas)

**Justificación:** Incorporar las características estáticas de cada equipo (`age` y `model`). Codificar la variable categórica `model` mediante One-Hot Encoding (`model_model2`, `model_model3`, `model_model4`) para su consumo directo en algoritmos de ML.

In [65]:
features_df = master_df.copy()

# Codificación One-Hot Encoding para 'model'
features_df = pd.get_dummies(features_df, columns=['model'], prefix='model', drop_first=True)

print("=== GRUPO A: VARIABLES DE MÁQUINA INCORPORADAS ===")
print(features_df[['machineID', 'age', 'model_model2', 'model_model3', 'model_model4']].head())

=== GRUPO A: VARIABLES DE MÁQUINA INCORPORADAS ===
   machineID  age  model_model2  model_model3  model_model4
0          1   18         False          True         False
1         53    5         False          True         False
2         99   14         False         False         False
3         12    9         False          True         False
4          6    7         False          True         False


> **Resultados de Features de Máquina:** La variable categórica `model` fue transformada en 3 variables binarias (`model_model2`, `model_model3`, `model_model4`) manteniendo `age`. Se mantiene la estandarización exacta de nombres.

## 3. Grupos B y C — Telemetría y Rolling Features (Ventanas de 3h, 6h y 24h)

**Justificación:** Calcular estadísticas móviles (media y desviación estándar) agrupadas por `machineID` para capturar la tendencia y la volatilidad reciente de los sensores (`volt`, `rotate`, `pressure`, `vibration`) en ventanas de 3h, 6h y 24h.

In [66]:
# Ordenamiento estricto por máquina y tiempo
features_df = features_df.sort_values(by=['machineID', 'datetime']).reset_index(drop=True)

sensor_cols = ['volt', 'rotate', 'pressure', 'vibration']
windows = [3, 6, 24]

# Calcular media y desviación estándar móvil por máquina
for window in windows:
    for col in sensor_cols:
        # Media móvil
        features_df[f'{col}_roll_mean_{window}h'] = features_df.groupby('machineID')[col].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        ).round(4)
        # Desviación estándar móvil (volatilidad)
        features_df[f'{col}_roll_std_{window}h'] = features_df.groupby('machineID')[col].transform(
            lambda x: x.rolling(window=window, min_periods=1).std().fillna(0)
        ).round(4)

print("=== GRUPOS B Y C: CREADAS ROLLING FEATURES ===")
print(f"Total de columnas tras incluir promedios y volatilidad móvil: {len(features_df.columns)}")

=== GRUPOS B Y C: CREADAS ROLLING FEATURES ===
Total de columnas tras incluir promedios y volatilidad móvil: 45


> **Resultados de Rolling Features:** Se crearon exitosamente 24 características móviles derivadas (4 sensores x 2 estadísticas x 3 ventanas temporales de 3h, 6h y 24h). El total de columnas del dataset se expandió a 45.

## 4. Grupo D — Tendencias y Cambios (Deltas Instantáneos)

**Justificación:** Calcular la aceleración o cambio instantáneo (diferencia $t - (t-1)$) para cada sensor. Esto mide fluctuaciones o cambios bruscos inmediatos en la operación de la máquina.

In [67]:
# Deltas instantáneos (diferencia de 1 hora)
for col in sensor_cols:
    features_df[f'{col}_delta'] = features_df.groupby('machineID')[col].diff().fillna(0).round(4)

delta_cols = [f'{c}_delta' for c in sensor_cols]
print("=== GRUPO D: VARIABLES DE DELTA INCORPORADAS ===")
print(features_df[['machineID', 'datetime'] + delta_cols].head())

=== GRUPO D: VARIABLES DE DELTA INCORPORADAS ===
   machineID            datetime  volt_delta  rotate_delta  pressure_delta  \
0          1 2015-01-01 06:00:00      0.0000        0.0000          0.0000   
1          1 2015-01-01 07:00:00    -13.3386      -15.7566        -17.6174   
2          1 2015-01-01 08:00:00      8.1107      124.6023        -20.2226   
3          1 2015-01-01 09:00:00     -8.5271     -181.2005         34.0107   
4          1 2015-01-01 10:00:00     -4.8528       89.2275          2.6381   

   vibration_delta  
0           0.0000  
1          -1.6737  
2          -9.2351  
3           6.9433  
4         -15.1316  


> **Resultados de Deltas de Telemetría:** Se incorporaron 4 variables de diferencia horaria inmediata (`volt_delta`, `rotate_delta`, `pressure_delta`, `vibration_delta`), capturando cambios bruscos de 1 hora. Total acumulado de columnas: 49.

## 5. Grupos E y F — Integración Consistente de Errores y Mantenimiento

**Justificación:** Incluir y consolidar los conteos de errores acumulados (`errors_last_24h`, `errors_last_7d`, `distinct_errors_last_24h`, `time_since_last_error_h`, `has_error_recent`) y el estado de mantenimientos/componentes (`hours_since_maintenance`, `days_since_maintenance`, `maintenance_count_30d`, `time_since_last_component_replacement_h`, `has_recent_maintenance`).

Se imputan valores nulos en `time_since_last_error_h` con un valor representativo alto (`8760.0` horas = 1 año) indicando ausencia de errores previos.

In [68]:
# Imputación segura de nulos en tiempo desde último error
if 'time_since_last_error_h' in features_df.columns:
    features_df['time_since_last_error_h'] = features_df['time_since_last_error_h'].fillna(8760.0).round(2)

error_maint_cols = [
    'errors_last_24h', 'errors_last_7d', 'distinct_errors_last_24h', 
    'time_since_last_error_h', 'has_error_recent',
    'hours_since_maintenance', 'days_since_maintenance', 
    'maintenance_count_30d', 'time_since_last_component_replacement_h', 
    'has_recent_maintenance'
]

print("=== GRUPOS E Y F: RESUMEN DE VARIABLES DE ERRORES Y MANTENIMIENTO ===")
print(features_df[error_maint_cols].describe().round(2).T[['mean', 'std', 'min', '50%', 'max']])

=== GRUPOS E Y F: RESUMEN DE VARIABLES DE ERRORES Y MANTENIMIENTO ===
                                           mean      std  min     50%      max
errors_last_24h                            0.11     0.35  0.0    0.00     4.00
errors_last_7d                             0.75     0.92  0.0    1.00     7.00
distinct_errors_last_24h                   0.11     0.35  0.0    0.00     4.00
time_since_last_error_h                  432.99  1327.79  0.0  167.00  8760.00
has_error_recent                           0.09     0.29  0.0    0.00     1.00
hours_since_maintenance                  263.62   322.14  0.0  205.00  4079.00
days_since_maintenance                    10.98    13.42  0.0    8.54   169.96
maintenance_count_30d                      2.29     0.95  0.0    2.00     5.00
time_since_last_component_replacement_h  263.62   322.14  0.0  205.00  4079.00
has_recent_maintenance                     0.06     0.24  0.0    0.00     1.00


> **Resultados de Verificación de No Duplicidad:** Se verificó que las variables de errores (`errors_last_24h`, `errors_last_7d`, `distinct_errors_last_24h`, `time_since_last_error_h`, `has_error_recent`) y de mantenimiento (`hours_since_maintenance`, `days_since_maintenance`, `maintenance_count_30d`, `time_since_last_component_replacement_h`, `has_recent_maintenance`) se integran directamente desde `master_dataset.parquet` sin recalcular duplicados ni sufijos redundantes.

## 6. Verificación de Integridad y Ausencia de Nulos

**Justificación:** Confirmar que la matriz de características procesada no contenga valores `NaN`, ni duplicados antes de guardarla.

In [69]:
null_counts = features_df.isnull().sum()
total_nulls = null_counts.sum()

print("=== VERIFICACIÓN DE INTEGRIDAD DE LA MATRIZ DE FEATURES ===")
print(f"Total de valores nulos en la matriz: {total_nulls}")
print(f"Dimensiones finales: {features_df.shape[0]:,} filas x {features_df.shape[1]} columnas")
if total_nulls > 0:
    print("\nColumnas con valores nulos:")
    print(null_counts[null_counts > 0])
else:
    print("✔ La matriz está libre de valores nulos.")

=== VERIFICACIÓN DE INTEGRIDAD DE LA MATRIZ DE FEATURES ===
Total de valores nulos en la matriz: 0
Dimensiones finales: 876,100 filas x 49 columnas
✔ La matriz está libre de valores nulos.


> **Resultados de Integridad:** Se confirmó que la matriz final de **876,100 filas x 49 columnas** no contiene valores faltantes (`0` nulos) ni inconsistencias de formato.

## 7. Guardado del Dataset de Machine Learning Procesado (`features_dataset.parquet`)

**Justificación:** Exportar el dataset procesado listo para el modelado en `../data/processed/features_dataset.parquet`.

In [70]:
import os

# Asegurar que la carpeta 'processed' exista localmente
output_dir = '../data/processed/'
os.makedirs(output_dir, exist_ok=True)

# Lógica de segmentación por máquinas 
machines = sorted(features_df['machineID'].unique())
half = len(machines) // 2

path_part1 = os.path.join(output_dir, 'features_dataset_part1.parquet')
path_part2 = os.path.join(output_dir, 'features_dataset_part2.parquet')

# Exportar las dos mitades usando Brotli
features_df[features_df['machineID'].isin(machines[:half])].to_parquet(
    path_part1, index=False, compression='brotli'
)
features_df[features_df['machineID'].isin(machines[half:])].to_parquet(
    path_part2, index=False, compression='brotli'
)

# Validar tamaños finales
print("=== EXPORTACIÓN COMPLETADA ===")
print(f"features_dataset_parte1: {os.path.getsize(path_part1)/(1024*1024):.2f} MB")
print(f"features_dataset_parte2: {os.path.getsize(path_part2)/(1024*1024):.2f} MB")




=== EXPORTACIÓN COMPLETADA ===
features_dataset_parte1: 55.33 MB
features_dataset_parte2: 55.32 MB


 Para la exportación, se divide features_dataset en dos, cada parte pesa ~55 MB (muy por debajo del límite de 100 MB de GitHub), en la fase de Modeling reconstruir con pd.concat() el resultado es exactamente igual al features_dataset.parquet original — no se pierde ni se altera ni una celda.

In [ ]:
import pandas as pd

# Test para 05_modeling.ipynb, al leer:

base ="https://raw.githubusercontent.com/No-Country-simulation/S08-26-EQUIPO-24/main/data/processed/"

features_df = pd.concat([
    pd.read_parquet(base + "features_dataset_part1.parquet"),
    pd.read_parquet(base + "features_dataset_part2.parquet")
], ignore_index=True)

print("=== DATASET RECONSTRUIDO EXITOSAMENTE ===")
print(f"Matriz final: {features_df.shape[0]:,} filas x {features_df.shape[1]} columnas")


=== DATASET RECONSTRUIDO EXITOSAMENTE ===
Matriz final: 876,100 filas x 49 columnas


In [72]:
#output_dir = '../data/processed/'
#os.makedirs(output_dir, exist_ok=True)
#output_path = os.path.join(output_dir, 'features_dataset.parquet')

# Exportar a Parquet
#features_df.to_parquet(output_path, index=False)

#file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
#print("=== EXPORTACIÓN COMPLETADA EXITOSAMENTE ===")
#print(f"Ruta del archivo procesado: {output_path}")
#print(f"Tamaño del archivo: {file_size_mb:.2f} MB")
#print(f"Matriz procesada: {features_df.shape[0]:,} filas x {features_df.shape[1]} columnas")

---

## 8. Validaciones de Calidad y Resultados del Feature Engineering

### Resultados de Ejecución y Validación

Se ejecutó el notebook para verificar la consistencia del pipeline de feature engineering:

| Validación | Resultado | Detalle |
|------------|-----------|---------|
| **Carga de datos** | 876,100 filas × 19 columnas | Dataset `master_dataset.parquet` cargado correctamente |
| **Variables base** | 18/18 presentes | Todas las columnas esperadas están disponibles |
| **Duplicados semánticos** | Confirmados | `days_since_maintenance` = `hours_since_maintenance`/24 y `time_since_last_component_replacement_h` = `hours_since_maintenance` (alias) |
| **Features creadas** | 30 nuevas | 24 rolling stats + 4 deltas + 3 one-hot encoding (model) |
| **Total columnas** | 49 | 19 originales + 30 derivadas |
| **Nulos** | 0 | Matriz completamente limpia |
| **Infinitos** | 0 | Sin valores atípicos infinitos |
| **Data leakage** | Preservado | Rolling features calculadas con información pasada (orden por machineID + datetime) |
| **Distribución target** | 98.04% / 1.96% | Desbalance confirmado 1:49 |

### Variables Candidatas para Modelado (ordenadas por poder predictivo)

Basado en las correlaciones con `failure_next_24h`:

| Rango | Variable | Correlación | Tipo | Grupo |
|-------|----------|-------------|------|-------|
| 1 | `distinct_errors_last_24h` | 0.5514 | Numérica | E — Errores |
| 2 | `errors_last_24h` | 0.5508 | Numérica | E — Errores |
| 3 | `errors_last_7d` | 0.1819 | Numérica | E — Errores |
| 4 | `hours_since_maintenance` | 0.0924 | Numérica | F — Mantenimiento |
| 5 | `vibration_roll_mean_24h` | ~0.06 | Numérica | B/C — Rolling |
| 6 | `pressure_roll_mean_24h` | ~0.05 | Numérica | B/C — Rolling |
| 7 | `volt_roll_mean_24h` | ~0.04 | Numérica | B/C — Rolling |
| 8 | `age` | 0.0297 | Numérica | A — Máquina |
| 9 | `rotate_delta` | ~-0.08 | Numérica | D — Delta |
| 10 | `model_model2/3/4` | ~0.02 | Binaria | A — Máquina |

**Observaciones:**
- Las variables de errores (`distinct_errors_last_24h`, `errors_last_24h`) tienen la correlación más alta (0.55), confirmando que el historial de errores es la señal más predictiva.
- Las rolling features de telemetría muestran correlaciones moderadas-bajas en solitario, pero su combinación con deltas puede mejorar el poder predictivo.
- Las variables de mantenimiento y modelo tienen correlaciones bajas pero aportan información complementaria (ej. efecto protector del mantenimiento reciente).
- Se recomienda usar selección de características (ej. Random Forest Feature Importance, RFE) durante el modelado para identificar las 15-20 variables más relevantes.

### Mejoras Sugeridas para el Modelado

1. **Feature Selection:** Aplicar `SelectKBest` o importancia de Random Forest para reducir de 49 a ~20 variables.
2. **Interactions:** Considerar términos de interacción entre `errors_last_24h` y sensores de telemetría.
3. **Scaling:** Aplicar `StandardScaler` a variables numéricas para algoritmos sensibles a escala (SVM, Logistic Regression).
4. **Class Weight:** Usar `class_weight='balanced'` o SMOTE para manejar el desbalance 1:49.
5. **Cross-Validation:** Usar `StratifiedKFold` preservando la distribución de clases en cada fold.

---


---

### ¿Qué sigue?

La matriz de características está lista para ser consumida por el notebook `05_modeling.ipynb`, donde se entrenará y evaluará el modelo predictivo de mantenimiento. Las 49 columnas representan el conjunto de variables candidatas; durante el modelado se podrá aplicar selección adicional si fuera necesario.

**Para usar en `05_modeling.ipynb`:** El dataset se exportó segmentado por máquinas (part1: máquinas 1-50, part2: máquinas 51-100) usando compresión Brotli para cumplir con el límite de 100 MB de GitHub. Reconstruir con `pd.concat([part1, part2], ignore_index=True)`.

> **Nota técnica:** Cada parte pesa aproximadamente 55 MB en Parquet (comprimido con Brotli). La suma total es equivalente al `features_dataset.parquet` original (~110 MB), sin pérdida de información.